# 16 — Per-(model, species) Isotonic Calibrators

Step 2 of the deploy pipeline (XGBoost ✅ → **calibrators** → submission).

**Input:** fold-0 OOFs for every base model — NN models from `experiments/{run}/fold0/oof_preds.csv` (234 species cols), XGBoost from `experiments/classical_xgboost/oof_preds.csv` (206 species cols, subset of the NN set).

**Output:** `experiments/calibrators/isotonic_calibrators.pkl`, a `dict[(model_name, species_code), IsotonicRegression]`. Species below `min_pos=2` or absent from a model's OOF are simply not in the dict — inference falls back to identity remap on missing keys.

**Validation target:** apply the saved calibrators back to fold-0 OOFs, mean-blend, confirm macro AUC reproduces **~0.983**. Below 0.980 ⇒ alignment or serialization bug, stop and investigate.

The canonical species set is taken from the **NN OOF column headers** (expected 234 cols, matches the BirdCLEF26 submission format), not from `train.csv` which only has 206 primary_labels in this dataset.

## 0 — Setup

In [ ]:
import os, pickle, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

ROOT     = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
META_CSV = ROOT / 'data' / 'raw' / 'train.csv'
EXP_DIR  = ROOT / 'experiments'

XGB_OOF    = EXP_DIR / 'classical_xgboost' / 'oof_preds.csv'
CALIB_DIR  = EXP_DIR / 'calibrators'
CALIB_PATH = CALIB_DIR / 'isotonic_calibrators.pkl'

VAL_FOLD = 0
MIN_POS  = 2

# Same mapping as notebook 11. Edit to match your real paths.
NN_MODELS = {
    'effb0':        'baseline_effb0/baseline_effb0',
    'effv2s':       'effv2s_finetune/effv2s_finetune',
    'convnext':     'sed_finetune/sed_finetune',
    'seresnext':    'seresnext_finetune/seresnext_finetune',
    'effv2s_focal': 'effvs2_focal/effv2s_focal',
}
XGB_NAME = 'xgboost'

print(f'NN OOF roots tried: {list(NN_MODELS.keys())}')
print(f'XGB OOF path      : {XGB_OOF}  (exists: {XGB_OOF.exists()})')
print(f'calibrator output : {CALIB_PATH}')

## 1 — Load fold-0 OOFs

Two naming patterns are tried per NN model (mirrors notebook 11's `load_oof`). XGBoost is a single CSV in `experiments/classical_xgboost/oof_preds.csv`. Any model that doesn't load is dropped with a warning.

In [ ]:
def load_fold0_oof(base_pattern, fold=VAL_FOLD):
    for pat in [f'{base_pattern}_fold{fold}/oof_preds.csv',
                f'{base_pattern}/fold{fold}/oof_preds.csv']:
        p = EXP_DIR / pat
        if p.exists():
            return pd.read_csv(p)
    return None


loaded = {}
for name, pattern in NN_MODELS.items():
    df = load_fold0_oof(pattern)
    if df is not None:
        loaded[name] = df
        print(f'  ✅ {name:<14} {len(df):>5} rows × {df.shape[1]-1} species cols')
    else:
        print(f'  ❌ {name:<14} MISSING — {pattern!r} fold {VAL_FOLD} not found')

if XGB_OOF.exists():
    loaded[XGB_NAME] = pd.read_csv(XGB_OOF)
    print(f'  ✅ {XGB_NAME:<14} {len(loaded[XGB_NAME]):>5} rows × {loaded[XGB_NAME].shape[1]-1} species cols')
else:
    raise FileNotFoundError(f'XGBoost OOF missing at {XGB_OOF} — run notebook 15 first')

assert len(loaded) >= 2, f'need ≥2 base models, got {len(loaded)}'
print(f'\n{len(loaded)} base models loaded')

## 2 — Canonical species set + per-model availability

Canonical column set = the one held by the most models (typically the 234-col NN format). The XGBoost OOF is a subset; track which canonical species each model has predictions for. Anything weirder (e.g. an OOF with completely different column names) is dropped with a warning rather than auto-fixed — re-export it cleanly.

In [ ]:
# Canonical species set — most-common column layout wins
set_counts = Counter(frozenset(c for c in df.columns if c != 'filename') for df in loaded.values())
canonical_frozen, votes = set_counts.most_common(1)[0]
canonical_set = set(canonical_frozen)
species_cols  = sorted(canonical_set)
n_species     = len(species_cols)
print(f'canonical species set: {n_species} cols  (held by {votes}/{len(loaded)} models)')

# Per-model: which canonical species does this model have predictions for?
model_species_mask = {}
keep_models = []
for name, df in loaded.items():
    cols = set(df.columns) - {'filename'}
    if cols == canonical_set:
        n_avail = n_species
    elif cols.issubset(canonical_set):
        n_avail = len(cols)
    elif canonical_set.issubset(cols):
        # Superset (e.g. dual-head export) — drop extras
        cols = cols & canonical_set
        n_avail = n_species
        print(f'  ⚠️  {name}: superset detected, dropping {len(set(df.columns) - {"filename"}) - n_avail} extra cols')
    else:
        overlap = len(cols & canonical_set)
        print(f'  ❌ {name}: {len(cols)} cols, only {overlap}/{n_species} overlap with canonical — DROPPING')
        continue
    mask = np.array([c in cols for c in species_cols])
    model_species_mask[name] = mask
    keep_models.append(name)
    extra = '' if n_avail == n_species else f'  (missing {n_species - n_avail} species — will identity-remap at inference)'
    print(f'  {name:<14} {n_avail}/{n_species} canonical species{extra}')

loaded = {name: loaded[name] for name in keep_models}
print(f'\nproceeding with {len(loaded)} models: {list(loaded.keys())}')

## 3 — Align rows and build labels

Intersect filenames across all surviving OOFs (matches notebook 11's pattern). Build a per-model `(N, n_species)` prediction matrix with `0.5` filled for canonical species the model doesn't have. Labels come from `train.csv.primary_label`. Note: species absent from `train.csv` will have 0 positives → automatically skipped by both the calibrator fit (`min_pos=2`) and the macro-AUC metric.

In [ ]:
meta = pd.read_csv(META_CSV)
common_files = set.intersection(*(set(df['filename']) for df in loaded.values()))
filenames = sorted(common_files)
N = len(filenames)
print(f'common fold-0 files across {len(loaded)} OOFs: {N}')

# Per-model aligned prediction matrices, shape (N, n_species), 0.5 for missing species
aligned = {}
for name, df in loaded.items():
    df_idx = df.set_index('filename').loc[filenames]
    avail_cols = [c for c in species_cols if c in df.columns]
    p = np.full((N, n_species), 0.5, dtype=np.float32)
    if avail_cols:
        col_pos = [species_cols.index(c) for c in avail_cols]
        p[:, col_pos] = df_idx[avail_cols].values.astype(np.float32)
    aligned[name] = p

# Label matrix from primary_label
merged = pd.DataFrame({'filename': filenames}).merge(
    meta[['filename', 'primary_label']], on='filename', how='left')
y_true = np.zeros((N, n_species), dtype=np.float32)
for i, sp in enumerate(species_cols):
    y_true[:, i] = (merged['primary_label'] == sp).astype(float)

pos_per_class = y_true.sum(axis=0)
n_with_pos    = int((pos_per_class > 0).sum())
n_fittable    = int((pos_per_class >= MIN_POS).sum())
print(f'positives/class: min={pos_per_class.min():.0f}  median={int(np.median(pos_per_class))}  max={pos_per_class.max():.0f}')
print(f'classes with ≥1 positive: {n_with_pos}/{n_species}  (the {n_species - n_with_pos} with 0 positives are skipped by metric & calibrator alike)')
print(f'classes with ≥{MIN_POS} positives (fittable): {n_fittable}/{n_species}')

## 4 — Fit calibrators

For each `(model, species)`: fit `IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1)` iff the model has predictions for that species **and** there are ≥2 positives in fold-0 labels. Skipped pairs are simply absent from the dict.

Runtime: ~30s for ~1,000 calibrators on a workstation.

In [ ]:
def fit_calibrators(aligned, y_true, species_cols, model_species_mask, min_pos=MIN_POS):
    calibrators = {}
    skip = Counter()
    for name, p_stack in aligned.items():
        avail = model_species_mask[name]
        for c, sp in enumerate(species_cols):
            if not avail[c]:
                skip[(name, 'no-pred')] += 1
                continue
            yc = y_true[:, c]
            n_pos = int(yc.sum())
            if n_pos < min_pos or n_pos > len(yc) - min_pos:
                skip[(name, 'low-pos')] += 1
                continue
            iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
            iso.fit(p_stack[:, c], yc)
            calibrators[(name, sp)] = iso
    return calibrators, skip


import time
t0 = time.time()
calibrators, skip_counts = fit_calibrators(aligned, y_true, species_cols, model_species_mask)
print(f'fitted {len(calibrators)} calibrators in {time.time()-t0:.1f}s')

print('\nPer-model breakdown:')
for name in aligned:
    fitted   = sum(1 for (m, _) in calibrators if m == name)
    no_pred  = skip_counts.get((name, 'no-pred'), 0)
    low_pos  = skip_counts.get((name, 'low-pos'), 0)
    print(f'  {name:<14} fitted={fitted:>3}  no-pred={no_pred:>3}  low-pos={low_pos:>3}  '
          f'total={fitted+no_pred+low_pos}')

In [ ]:
# Save the dict per the user spec: dict[(model_name, species_code) -> IsotonicRegression]
CALIB_DIR.mkdir(parents=True, exist_ok=True)
with open(CALIB_PATH, 'wb') as f:
    pickle.dump(calibrators, f)
size_mb = CALIB_PATH.stat().st_size / (1024**2)
print(f'saved → {CALIB_PATH}  ({size_mb:.1f} MB, {len(calibrators)} entries)')

# Round-trip load to catch pickle issues immediately
with open(CALIB_PATH, 'rb') as f:
    loaded_calib = pickle.load(f)
assert len(loaded_calib) == len(calibrators)
assert all(isinstance(v, IsotonicRegression) for v in loaded_calib.values())
# Spot-check: pick one entry, ensure predict gives the same output
some_key = next(iter(calibrators))
x_test = np.array([0.1, 0.5, 0.9], dtype=np.float32)
np.testing.assert_array_equal(calibrators[some_key].predict(x_test), loaded_calib[some_key].predict(x_test))
print('round-trip load + predict OK')

## 5 — Validate: apply calibrators and reproduce the blend

Apply the saved calibrators back to the fold-0 OOFs (the same data they were fit on — this is the deployment-config self-test, not a held-out evaluation) and check the calibrated mean blend.

**Note on AUC interpretation.** Isotonic preserves rank order (it's monotonic non-decreasing) but introduces ties via its flat regions, so calibrated AUC may be marginally higher than raw AUC on training data — typically within 0.005 for well-fit models, more for poorly-calibrated ones. The point of calibration isn't per-model AUC; it's making per-model probabilities *commensurable* so the **mean blend** doesn't get hijacked by any single model's miscalibration. The blend AUC is what we validate against the 0.983 target.

**Targets to confirm:**
- Calibrated mean (all base models): **~0.983** ← gate at 0.980
- Calibrated mean (NN only, no XGB): **~0.976** ← if both numbers reproduce, alignment is correct
- Lift from adding XGBoost: **+0.0067** (or whatever's between the two)

In [ ]:
def macro_auc_skip_empty(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan'), len(aucs)


def apply_calibrators(aligned, calibrators, species_cols):
    """Apply per-(model, species) isotonic. Identity remap when no calibrator."""
    out = {}
    for name, p_stack in aligned.items():
        c_out = p_stack.copy().astype(np.float32)
        for c, sp in enumerate(species_cols):
            iso = calibrators.get((name, sp))
            if iso is not None:
                c_out[:, c] = iso.predict(p_stack[:, c]).astype(np.float32)
        out[name] = c_out
    return out


calibrated = apply_calibrators(aligned, loaded_calib, species_cols)

print('Per-model fold-0 macro AUC (raw vs calibrated — small lift on training data is expected):')
for name in aligned:
    raw, _ = macro_auc_skip_empty(y_true, aligned[name])
    cal, _ = macro_auc_skip_empty(y_true, calibrated[name])
    print(f'  {name:<14}  raw={raw:.4f}  calibrated={cal:.4f}  Δ={cal-raw:+.4f}')

In [ ]:
# Blend AUCs across various subsets so you can identify which one produced the 0.976/0.983 reference
nn_names = [n for n in aligned if n != XGB_NAME]
nn4_names = [n for n in nn_names if n != 'effv2s_focal']   # the 4-NN subset, if applicable

def blend_auc(names, source):
    if not names: return None, 0
    stack = np.stack([source[n] for n in names], axis=0)
    auc, _ = macro_auc_skip_empty(y_true, stack.mean(0))
    return auc, len(names)

results = []
for label, source in [('raw', aligned), ('calibrated', calibrated)]:
    auc_all, n_all = blend_auc(list(aligned.keys()), source)
    auc_nn,  n_nn  = blend_auc(nn_names, source)
    auc_nn4, n_nn4 = blend_auc(nn4_names, source) if nn4_names else (None, 0)
    results.append((label, auc_all, n_all, auc_nn, n_nn, auc_nn4, n_nn4))

print(f'{"blend":<16} {"all models":>16} {"NN only":>16} {"4-NN subset":>16}')
for label, a, na, n, nn, n4, nn4 in results:
    a_s  = f'{a:.4f} ({na})'  if a  is not None else 'n/a'
    n_s  = f'{n:.4f} ({nn})'  if n  is not None else 'n/a'
    n4_s = f'{n4:.4f} ({nn4})' if n4 is not None else 'n/a'
    print(f'  {label:<14} {a_s:>16} {n_s:>16} {n4_s:>16}')

# Pick the validation gate based on what's present
auc_cal_all = results[1][1]   # calibrated, all models
auc_cal_nn  = results[1][3]   # calibrated, NN only
print(f'\nlift from XGBoost (calibrated):  all-vs-NN = {auc_cal_all - auc_cal_nn:+.4f}')

GATE = 0.980
if auc_cal_all >= GATE:
    print(f'\n✅ calibrated all-model blend {auc_cal_all:.4f} ≥ {GATE} — proceed to step 3')
else:
    print(f'\n⚠️  calibrated all-model blend {auc_cal_all:.4f} < {GATE} — investigate before step 3')

## 6 — Handoff to step 3

**Deliverables from this notebook:**
- `experiments/calibrators/isotonic_calibrators.pkl` — `dict[(model_name, species_code), IsotonicRegression]`

**Inference contract for step 3:**
```python
import pickle
with open('isotonic_calibrators.pkl', 'rb') as f:
    calibrators = pickle.load(f)

# For each model's per-clip predictions (N, n_species), in canonical species order:
for c, species_code in enumerate(species_cols):
    key = (model_name, species_code)
    if key in calibrators:
        calibrated[:, c] = calibrators[key].predict(raw[:, c])
    # else: identity (raw values pass through unchanged)

# Mean-blend the 5 calibrated matrices, write submission CSV.
```

**Kaggle dataset upload:**
- 5 ONNX base models  (from `scripts/export_onnx.py`)
- `xgboost_deploy.pkl`        (70.7 MB from step 1)
- `isotonic_calibrators.pkl`  (size printed above)

**Step 3 plan (next notebook):** modify the existing inference notebook in `submissions/` to extract the 620-dim classical feature vector alongside the mel spec, predict from all 5 base models, apply the loaded calibrators, mean-blend, and write the submission CSV in the BirdCLEF26 format. End-to-end validation: run the pipeline on fold-0 val audio files, confirm macro AUC reproduces the calibrated blend AUC printed above.